<a href="https://colab.research.google.com/github/UXI-create/-_First-Project/blob/main/%E6%97%85%E9%81%8A%E8%A8%88%E7%95%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 🛑 第一步：環境準備與靜音
# ==========================================
from IPython.utils import io
import os
import warnings
import base64

with io.capture_output() as captured:
    !pip install -q geopy pandas ipywidgets requests

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import ipywidgets as widgets
import pandas as pd
from datetime import datetime, timedelta, date
from IPython.display import display, clear_output, HTML
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests

# ==========================================
# 🎨 視覺美化設定
# ==========================================
app_header = widgets.HTML("""
<div style="background: linear-gradient(135deg, #11998e, #38ef7d); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
    <h1 style="margin: 0; font-size: 28px; font-weight: bold; color: white;">✈️ Travel Vibe Pro Max</h1>
    <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">多重行程建檔 ✕ 雲端記憶庫 ✕ 完整規劃</p>
</div>
""")

def style_dataframe(df):
    return df.style.set_table_styles([
        {'selector': 'thead th', 'props': [('background-color', '#11998e'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding', '10px')]},
        {'selector': 'tbody td', 'props': [('text-align', 'center'), ('padding', '8px'), ('border-bottom', '1px solid #eeeeee')]},
        {'selector': 'tbody tr:hover', 'props': [('background-color', '#eafff5')]}
    ]).hide(axis="index").format({'💰 預算': 'NT$ {:,.0f}'})

# ==========================================
# 💾 微型資料庫 (存放多個行程)
# ==========================================
trip_database = {}

# 全域 UI 樣式設定 (解決擠壓問題)
ui_style = {'description_width': 'initial'}

# ==========================================
# ⛅ 模組一：智慧天氣與打包引擎
# ==========================================
visa_rules = {
    "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
    "🇭🇰 香港": "需申請『預辦入境登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
}

def get_weather(country_name):
    try:
        geolocator = Nominatim(user_agent="vibe_weather_app")
        loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
        if not loc: return None
        url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
        res = requests.get(url).json()
        return res['current_weather']
    except: return None

def generate_smart_packing_list(days, country):
    packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
    if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
    else:
        packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])
        if "中國" in country: packing["🪪 證件與金錢"].append("📱 支付寶/微信支付")
        else: packing["🪪 證件與金錢"].append("外幣現金 / 海外信用卡")

    weather_data = get_weather(country)
    if weather_data:
        temp, code = weather_data['temperature'], weather_data['weathercode']
        is_raining = code >= 50
        packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
        if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
        elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
        if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])

    packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
    packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
    return packing

# ==========================================
# 🗓️ 模組二：自動估算與跨日追蹤引擎
# ==========================================
class SmartPlanner:
    def __init__(self):
        self.itinerary = []
        self.current_time = datetime.now()
        self.geolocator = Nominatim(user_agent="travel_vibe_ultimate_router")
        self.search_cache = {}

    def search_locations(self, city, keyword):
        query = f"{keyword}, {city}" if city else keyword
        try:
            locs = self.geolocator.geocode(query, exactly_one=False, limit=5)
            if not locs: locs = self.geolocator.geocode(keyword, exactly_one=False, limit=5)
            results = {}
            if locs:
                for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
            return results
        except: return {}

    def estimate_time(self, loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(self, mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    def add_stop(self, place_address, stay_mins, trans_mode, cost, note, start_datetime=None):
        place_data = self.search_cache.get(place_address)
        if not place_data: return False, "系統錯誤"

        stay_time = timedelta(minutes=stay_mins)
        if not self.itinerary:
            if start_datetime: self.current_time = start_datetime
            arrival_time = self.current_time
            transport_text = "✨ 出發"
        else:
            last_stop = self.itinerary[-1]
            trans_mins = self.estimate_time(last_stop['coords'], place_data, trans_mode)
            arrival_time = self.current_time + timedelta(minutes=trans_mins)
            transport_text = f"{trans_mode} (約 {self.format_time_str(trans_mins)})"

        departure_time = arrival_time + stay_time

        self.itinerary.append({
            "seq": len(self.itinerary) + 1,
            "arrive": arrival_time.strftime("%m/%d %H:%M"),
            "transport": transport_text,
            "name": place_data['name'],
            "note": note,
            "stay": self.format_time_str(stay_mins),
            "cost": cost,
            "depart": departure_time.strftime("%m/%d %H:%M"),
            "coords": place_data
        })
        self.current_time = departure_time
        return True, ""

    def reset(self):
        self.itinerary = []

# ==========================================
# 📱 模組三：UI 介面 (修復擠壓問題)
# ==========================================
box_layout = widgets.Layout(padding='15px', border='1px solid #e0e0e0', border_radius='10px', margin='10px 0')
highlight_box_layout = widgets.Layout(padding='15px', border='2px solid #11998e', border_radius='10px', margin='10px 0', background_color='#f4fff9')

# --- 行李分頁 ---
dest_dropdown = widgets.Dropdown(options=list(visa_rules.keys()), value='🇯🇵 日本', description='✈️ 目的地:', style=ui_style)
days_input = widgets.IntText(value=5, description='📅 天數:', layout=widgets.Layout(width='150px'), style=ui_style)
pack_btn = widgets.Button(description=" ⛅ 預測天氣並生成", button_style='success', layout=widgets.Layout(width='180px'))
pack_out = widgets.Output()

def on_pack_click(b):
    with pack_out:
        clear_output()
        print(f"📡 正在分析 {dest_dropdown.value} 當地氣候...")
        result = generate_smart_packing_list(days_input.value, dest_dropdown.value)
        clear_output()
        html_str = f"<h3 style='color: #11998e;'>🎒 【 {dest_dropdown.value} {days_input.value}天 】 必備清單</h3>"
        for cat, items in result.items():
            html_str += f"<div style='background: #f8f9fa; padding: 10px; border-left: 4px solid #38ef7d; margin-bottom: 10px;'>"
            html_str += f"<h4 style='margin-top: 0;'>{cat}</h4><ul style='list-style-type: none; padding-left: 10px;'>"
            for item in items: html_str += f"<li style='margin-bottom: 5px;'>🔲 {item}</li>"
            html_str += "</ul></div>"
        display(HTML(html_str))
pack_btn.on_click(on_pack_click)
tab_pack = widgets.VBox([widgets.HBox([dest_dropdown, days_input, pack_btn]), pack_out], layout=box_layout)

# --- 行程分頁 ---
planner = SmartPlanner()

# 檔案管理區 UI
trip_name_input = widgets.Text(description='📁 行程命名:', placeholder='例: 2024 東京跨年', layout=widgets.Layout(width='280px'), style=ui_style)
save_trip_btn = widgets.Button(description=" 💾 儲存目前行程", button_style='success')
saved_trips_dropdown = widgets.Dropdown(options=['無暫存紀錄'], description='📂 歷史紀錄:', layout=widgets.Layout(width='280px'), style=ui_style)
load_trip_btn = widgets.Button(description=" 📖 讀取", button_style='info')
new_trip_btn = widgets.Button(description=" 📄 開新空白行程", button_style='danger')
file_msg_out = widgets.Output()

# 🌟 這裡修復了時間選單被擠壓的問題
start_date_picker = widgets.DatePicker(description='📅 起點日期:', value=date.today(), layout=widgets.Layout(width='200px'), style=ui_style)
start_hour = widgets.Dropdown(options=[f"{i:02d}" for i in range(24)], value='08', description='時間:', layout=widgets.Layout(width='120px'), style=ui_style)
start_min = widgets.Dropdown(options=[f"{i:02d}" for i in range(0, 60, 5)], value='00', description='分:', layout=widgets.Layout(width='120px'), style=ui_style)

city_input = widgets.Text(value='東京', description='🌍 城市:', layout=widgets.Layout(width='180px'), style=ui_style)
keyword_input = widgets.Text(description='🔍 找景點/機場:', placeholder='例: 成田機場', layout=widgets.Layout(width='220px'), style=ui_style)
search_btn = widgets.Button(description=" 🔎 搜尋", button_style='info')

place_dropdown = widgets.Dropdown(options=['請先搜尋地點...'], description='📍 確認地點:', disabled=True, layout=widgets.Layout(width='450px'), style=ui_style)
stay_input = widgets.IntText(value=120, description='⏱️ 停留(分):', layout=widgets.Layout(width='160px'), style=ui_style)
trans_mode = widgets.Dropdown(options=['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車', '✈️ 搭飛機/高鐵'], value='🚌 公車/地鐵', description='交通:', layout=widgets.Layout(width='180px'), style=ui_style)

cost_input = widgets.IntText(value=0, description='💰 預算(元):', layout=widgets.Layout(width='160px'), style=ui_style)
note_input = widgets.Text(description='📝 備註:', placeholder='航班號 / 備註資訊', layout=widgets.Layout(width='320px'), style=ui_style)
add_btn = widgets.Button(description=" ➕ 加入行程", button_style='primary', disabled=True)
export_btn = widgets.Button(description=" 📥 匯出 CSV", button_style='success')
plan_out = widgets.Output()

def update_plan_view():
    with plan_out:
        clear_output()
        if not planner.itinerary:
            display(HTML("<p style='color: gray;'>📭 目前為空白行程。設定日期時間後，開始搜尋景點吧！</p>"))
            return

        df = pd.DataFrame(planner.itinerary)
        total_cost = df['cost'].sum()
        display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
        display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']

        display(style_dataframe(display_df))
        display(HTML(f"<h3 style='text-align: right; color: #11998e;'>💵 總花費: NT$ {total_cost:,.0f} </h3>"))

# 檔案管理邏輯
def on_save_click(b):
    with file_msg_out:
        clear_output()
        name = trip_name_input.value.strip()
        if not name:
            print("❌ 請先輸入行程名稱才能儲存！")
            return
        if not planner.itinerary:
            print("❌ 行程是空的，沒有東西可以存！")
            return
        import copy
        trip_database[name] = copy.deepcopy(planner.itinerary)
        saved_trips_dropdown.options = list(trip_database.keys())
        saved_trips_dropdown.value = name
        print(f"✅ 成功儲存行程：【{name}】！")

def on_load_click(b):
    with file_msg_out:
        clear_output()
        name = saved_trips_dropdown.value
        if name not in trip_database:
            print("❌ 找不到該行程紀錄。")
            return
        import copy
        planner.itinerary = copy.deepcopy(trip_database[name])
        start_date_picker.disabled = True
        start_hour.disabled = True
        start_min.disabled = True
        trip_name_input.value = name
        print(f"📖 成功讀取行程：【{name}】")
    update_plan_view()

def on_new_trip_click(b):
    planner.reset()
    trip_name_input.value = ""
    start_date_picker.disabled = False
    start_hour.disabled = False
    start_min.disabled = False
    with file_msg_out:
        clear_output()
        print("📄 已開啟全新空白行程，請重新設定出發日期！")
    update_plan_view()

save_trip_btn.on_click(on_save_click)
load_trip_btn.on_click(on_load_click)
new_trip_btn.on_click(on_new_trip_click)

def on_search_click(b):
    with plan_out:
        clear_output()
        display(HTML(f"<p>📡 雷達掃描中...</p>"))
    results = planner.search_locations(city_input.value, keyword_input.value)
    with plan_out:
        clear_output()
        if not results:
            place_dropdown.options = ['找不到地點']
            place_dropdown.disabled, add_btn.disabled = True, True
        else:
            planner.search_cache.update(results)
            place_dropdown.options = list(results.keys())
            place_dropdown.disabled, add_btn.disabled = False, False
            display(HTML(f"<p style='color: #27ae60;'>✅ 找到地點，請確認並填寫航班備註後點擊「加入」！</p>"))
            update_plan_view()

def on_add_click(b):
    if place_dropdown.disabled: return

    start_dt = None
    if not planner.itinerary:
        time_str = f"{start_hour.value}:{start_min.value}"
        start_dt = datetime.combine(start_date_picker.value, datetime.strptime(time_str, "%H:%M").time())

    planner.add_stop(place_dropdown.value, stay_input.value, trans_mode.value, cost_input.value, note_input.value, start_dt)

    keyword_input.value = ""
    cost_input.value = 0
    note_input.value = ""
    place_dropdown.options = ['請先搜尋下一個地點...']
    place_dropdown.disabled, add_btn.disabled = True, True
    start_date_picker.disabled = True
    start_hour.disabled = True
    start_min.disabled = True

    update_plan_view()

def on_export_click(b):
    if not planner.itinerary: return
    df = pd.DataFrame(planner.itinerary)
    export_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
    export_df.columns = ['抵達時間 (月/日 時:分)', '交通方式', '地點', '停留時間', '航班號/備註', '預估花費', '離開時間 (月/日 時:分)']

    csv_data = export_df.to_csv(index=False, encoding='utf-8-sig')
    b64 = base64.b64encode(csv_data.encode('utf-8-sig')).decode()

    trip_name = trip_name_input.value if trip_name_input.value else "未命名行程"
    href = f"""
    <div style="margin-top: 15px;">
        <a download="Travel_Vibe_{trip_name}.csv" href="data:text/csv;base64,{b64}" target="_blank"
           style="padding: 10px 20px; background-color: #11998e; color: white; text-decoration: none; border-radius: 5px; font-weight: bold; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">
           💾 點此下載【Travel_Vibe_{trip_name}.csv】
        </a>
    </div>
    """
    with plan_out:
         display(HTML(href))

search_btn.on_click(on_search_click)
add_btn.on_click(on_add_click)
export_btn.on_click(on_export_click)

# 佈局分組
file_row1 = widgets.HBox([trip_name_input, save_trip_btn])
file_row2 = widgets.HBox([saved_trips_dropdown, load_trip_btn, new_trip_btn])
file_section = widgets.VBox([widgets.HTML("<b>📂 歷史行程檔案管理</b>"), file_row1, file_row2, file_msg_out], layout=highlight_box_layout)

row0 = widgets.HBox([start_date_picker, start_hour, start_min])
row1 = widgets.HBox([city_input, keyword_input, search_btn])
row2 = widgets.HBox([place_dropdown])
row3 = widgets.HBox([stay_input, trans_mode, cost_input])
row4 = widgets.HBox([note_input, add_btn, export_btn])

tab_plan = widgets.VBox([
    file_section,
    widgets.HTML("<b>📍 STEP 1: 設定行程起點日期與時間</b>"), row0, widgets.HTML("<hr>"),
    widgets.HTML("<b>📍 STEP 2: 搜尋與加入行程</b>"), row1, row2, row3, row4,
    widgets.HTML("<hr>"), plan_out
], layout=box_layout)

app_tabs = widgets.Tab(children=[tab_plan, tab_pack])
app_tabs.set_title(0, '🗓️ 智慧行程與檔案庫')
app_tabs.set_title(1, '🧳 天氣與打包管家')

clear_output()
display(app_header)
display(app_tabs)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# 初始化網頁記憶體 (Session State)
if 'itinerary' not in st.session_state:
    st.session_state.itinerary = []
if 'trip_database' not in st.session_state:
    st.session_state.trip_database = {}
if 'search_results' not in st.session_state:
    st.session_state.search_results = {}
if 'current_time' not in st.session_state:
    st.session_state.current_time = datetime.now()

# ==========================================
# 🧠 核心邏輯引擎 (移植自原本的模組)
# ==========================================
geolocator = Nominatim(user_agent="vibe_streamlit_app")

visa_rules = {
    "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
    "🇭🇰 香港": "需申請『預辦入境登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
}

@st.cache_data # Streamlit 神器：快取抓過的天氣，避免重複連線
def get_weather(country_name):
    try:
        loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
        if not loc: return None
        url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
        res = requests.get(url).json()
        return res['current_weather']
    except: return None

def generate_smart_packing_list(days, country):
    packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
    if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
    else:
        packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])
        if "中國" in country: packing["🪪 證件與金錢"].append("📱 支付寶/微信支付")
        else: packing["🪪 證件與金錢"].append("外幣現金 / 海外信用卡")

    weather_data = get_weather(country)
    if weather_data:
        temp, code = weather_data['temperature'], weather_data['weathercode']
        is_raining = code >= 50
        packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
        if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
        elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
        if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])

    packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
    packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])

    if "台灣" not in country:
        if "港" in country or "澳" in country: packing["🔌 電子與雜物"].append("🔌 萬用轉接頭 (英規三腳方型)")
        elif "中國" in country: packing["🔌 電子與雜物"].extend(["🔌 轉接頭 (八字/雙扁)", "🌐 免翻牆網卡"])
        else: packing["🔌 電子與雜物"].extend(["🔌 萬用轉接頭", "🌐 網卡 / Wifi機"])
    return packing

def estimate_time(loc1, loc2, mode):
    if not loc1 or not loc2: return 20
    dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
    speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
    return max(3, int(((dist_km * 1.3) / speed) * 60))

def format_time_str(mins):
    if mins < 60: return f"{mins} 分"
    return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

# ==========================================
# 🎨 UI 介面設計 (Streamlit 版本)
# ==========================================
st.markdown("""
<div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
    <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ Travel Vibe Web App</h1>
    <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">真正的網頁版旅行管家！支援手機瀏覽</p>
</div>
""", unsafe_allow_html=True)

tab_plan, tab_pack = st.tabs(['🗓️ 智慧行程與檔案庫', '🧳 天氣與打包管家'])

# ----------------- 🧳 分頁：天氣與打包 -----------------
with tab_pack:
    col1, col2 = st.columns(2)
    with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
    with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)

    if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
        with st.spinner(f"正在連線 {dest} 氣象局..."):
            result = generate_smart_packing_list(days, dest)
            st.success("✅ 行李清單生成完畢！")

            for cat, items in result.items():
                st.markdown(f"#### {cat}")
                for item in items:
                    st.markdown(f"- [ ] {item}")
                st.markdown("---")

# ----------------- 🗓️ 分頁：行程與檔案庫 -----------------
with tab_plan:
    # 📂 檔案管理區
    with st.expander("📂 歷史行程檔案管理", expanded=False):
        c1, c2, c3 = st.columns([2, 1, 1])
        with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 2024 東京跨年")
        with c2:
            st.write("")
            if st.button("💾 儲存行程"):
                if trip_name and st.session_state.itinerary:
                    st.session_state.trip_database[trip_name] = copy.deepcopy(st.session_state.itinerary)
                    st.success(f"已儲存：{trip_name}")
                else:
                    st.error("請輸入名稱且行程不能為空")
        with c3:
            st.write("")
            if st.button("📄 開新空白行程"):
                st.session_state.itinerary = []
                st.rerun() # 重新整理網頁

        if st.session_state.trip_database:
            load_name = st.selectbox("📂 讀取歷史紀錄", list(st.session_state.trip_database.keys()))
            if st.button("📖 讀取此行程"):
                st.session_state.itinerary = copy.deepcopy(st.session_state.trip_database[load_name])
                st.success(f"已讀取：{load_name}")
                st.rerun()

    st.markdown("### 📍 新增行程節點")

    # 日期與時間 (如果是第一站才需要)
    is_first = len(st.session_state.itinerary) == 0
    if is_first:
        st.info("👋 這是第一站！請設定出發日期與時間。")
        c1, c2 = st.columns(2)
        with c1: start_date = st.date_input("📅 起點日期")
        with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

    # 搜尋區
    c1, c2, c3 = st.columns([1, 2, 1])
    with c1: city = st.text_input("🌍 城市", value="東京")
    with c2: keyword = st.text_input("🔍 找景點/機場", placeholder="例: 成田機場")
    with c3:
        st.write("") # 排版用
        if st.button("🔎 搜尋", type="primary"):
            with st.spinner("雷達掃描中..."):
                results = {}
                try:
                    query = f"{keyword}, {city}" if city else keyword
                    locs = geolocator.geocode(query, exactly_one=False, limit=5)
                    if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                    if locs:
                        for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                except: pass
                st.session_state.search_results = results

    # 選擇與加入區
    if st.session_state.search_results:
        selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))

        c1, c2, c3, c4 = st.columns(4)
        with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
        with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車', '✈️ 飛機/高鐵'], index=1)
        with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
        with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

        if st.button("➕ 確認加入行程", use_container_width=True):
            place_data = st.session_state.search_results[selected_address]
            stay_time = timedelta(minutes=stay)

            if is_first:
                st.session_state.current_time = datetime.combine(start_date, start_time)
                arrival_time = st.session_state.current_time
                transport_text = "✨ 出發"
            else:
                last_stop = st.session_state.itinerary[-1]
                trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

            departure_time = arrival_time + stay_time

            st.session_state.itinerary.append({
                "arrive": arrival_time.strftime("%m/%d %H:%M"),
                "transport": transport_text,
                "name": place_data['name'],
                "note": note,
                "stay": format_time_str(stay),
                "cost": cost,
                "depart": departure_time.strftime("%m/%d %H:%M"),
                "coords": place_data
            })
            st.session_state.current_time = departure_time
            st.session_state.search_results = {} # 清空搜尋結果
            st.rerun() # 重新載入畫面更新表格

    st.markdown("---")

    # 顯示行程表
    if st.session_state.itinerary:
        df = pd.DataFrame(st.session_state.itinerary)
        total_cost = df['cost'].sum()

        display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
        display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']

        # Streamlit 內建超美資料表
        st.dataframe(display_df, use_container_width=True, hide_index=True)
        st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總花費: NT$ {total_cost:,.0f} </h3>", unsafe_allow_html=True)

        # 內建一鍵下載 CSV (超級強大！)
        csv = display_df.to_csv(index=False, encoding='utf-8-sig').encode('utf-8-sig')
        st.download_button(
            label="📥 下載 Excel 格式行程表 (CSV)",
            data=csv,
            file_name=f"Travel_Vibe_行程表.csv",
            mime="text/csv",
            type="primary"
        )

In [ ]:
# 1. 安裝 Streamlit 與 Localtunnel 工具
!pip install -q streamlit
!npm install -g localtunnel > /dev/null

# 2. 獲取你的專屬「解鎖密碼」 (因為安全考量，連線網頁時會需要輸入)
import urllib
print("=========================================")
print("🔑 你的網站解鎖密碼是：", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("=========================================")

# 3. 在背景啟動 Travel Vibe Web App
!streamlit run app.py &>/dev/null &

# 4. 開啟任意門，產生公開網址！
!npx localtunnel --port 8501

In [ ]:
# 1. 安裝套件並清理舊的卡頓連線
!pip install -q streamlit
!pkill -f streamlit
!pkill -f localtunnel
!pkill -f cloudflared

# 2. 下載企業級 Cloudflare 通道工具
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 3. 在背景重新啟動你的 Travel Vibe App
!nohup streamlit run app.py &>/dev/null &

# 4. 啟動極速通道並擷取專屬網址
print("🚀 正在為您建立 Cloudflare 極速連線，請稍候約 5 秒鐘...")
!nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
!sleep 5

print("=========================================================")
print("✨ 你的極速版 App 網址已準備好，請點擊下方連結：")
!grep -o 'https://.*\.trycloudflare\.com' cloudflared.log | head -n 1
print("=========================================================")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# ==========================================
# 🔐 模組零：私有化登入系統 (Password Gateway)
# ==========================================
# 🌟 你的專屬密碼已經更新在這裡 👇
SECRET_PASSWORD = "201020"

def check_password():
    """回傳 True 如果使用者已經輸入正確密碼"""
    if "password_correct" not in st.session_state:
        st.session_state["password_correct"] = False

    if not st.session_state["password_correct"]:
        st.markdown("""
        <div style="background: linear-gradient(135deg, #1f4037, #99f2c8); padding: 30px; border-radius: 10px; text-align: center; color: white; margin-top: 50px;">
            <h1 style="margin: 0; font-size: 36px; font-weight: bold; color: white;">🔒 Travel Vibe 私有終端</h1>
            <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">請輸入通行憑證以解鎖您的專屬行程管家</p>
        </div>
        """, unsafe_allow_html=True)

        st.write("")

        password_input = st.text_input("🔑 通關密碼", type="password")

        if st.button("🔓 解鎖進入", use_container_width=True):
            if password_input == SECRET_PASSWORD:
                st.session_state["password_correct"] = True
                st.rerun()
            else:
                st.error("❌ 密碼錯誤，存取被拒絕！(Access Denied)")
        return False
    return True

if check_password():
    # ==========================================
    # 🧠 主程式邏輯與 UI
    # ==========================================
    if 'itinerary' not in st.session_state: st.session_state.itinerary = []
    if 'trip_database' not in st.session_state: st.session_state.trip_database = {}
    if 'search_results' not in st.session_state: st.session_state.search_results = {}
    if 'current_time' not in st.session_state: st.session_state.current_time = datetime.now()

    geolocator = Nominatim(user_agent="vibe_streamlit_app")

    visa_rules = {
        "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
        "🇭🇰 香港": "需申請『預辦登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
    }

    @st.cache_data
    def get_weather(country_name):
        try:
            loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
            if not loc: return None
            url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
            res = requests.get(url).json()
            return res['current_weather']
        except: return None

    def generate_smart_packing_list(days, country):
        packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
        if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
        else:
            packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])
            if "中國" in country: packing["🪪 證件與金錢"].append("📱 支付寶/微信支付")
            else: packing["🪪 證件與金錢"].append("外幣現金 / 海外信用卡")

        weather_data = get_weather(country)
        if weather_data:
            temp, code = weather_data['temperature'], weather_data['weathercode']
            is_raining = code >= 50
            packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
            if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
            elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
            if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])
        packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
        packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
        if "台灣" not in country:
            if "港" in country or "澳" in country: packing["🔌 電子與雜物"].append("🔌 萬用轉接頭 (英規三腳方型)")
            elif "中國" in country: packing["🔌 電子與雜物"].extend(["🔌 轉接頭 (八字/雙扁)", "🌐 免翻牆網卡"])
            else: packing["🔌 電子與雜物"].extend(["🔌 萬用轉接頭", "🌐 網卡 / Wifi機"])
        return packing

    def estimate_time(loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    st.markdown("""
    <div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
        <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ Travel Vibe Web App</h1>
        <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">私有化保密連線中 🔒</p>
    </div>
    """, unsafe_allow_html=True)

    if st.sidebar.button("🚪 登出系統"):
        st.session_state["password_correct"] = False
        st.rerun()

    tab_plan, tab_pack = st.tabs(['🗓️ 智慧行程與檔案庫', '🧳 天氣與打包管家'])

    with tab_pack:
        col1, col2 = st.columns(2)
        with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
        with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)

        if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
            with st.spinner(f"正在連線 {dest} 氣象局..."):
                result = generate_smart_packing_list(days, dest)
                st.success("✅ 行李清單生成完畢！")
                for cat, items in result.items():
                    st.markdown(f"#### {cat}")
                    for item in items: st.markdown(f"- [ ] {item}")
                    st.markdown("---")

    with tab_plan:
        with st.expander("📂 歷史行程檔案管理", expanded=False):
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 2024 東京跨年")
            with c2:
                st.write("")
                if st.button("💾 儲存行程"):
                    if trip_name and st.session_state.itinerary:
                        st.session_state.trip_database[trip_name] = copy.deepcopy(st.session_state.itinerary)
                        st.success(f"已儲存：{trip_name}")
                    else: st.error("請輸入名稱且行程不能為空")
            with c3:
                st.write("")
                if st.button("📄 開新空白行程"):
                    st.session_state.itinerary = []
                    st.rerun()
            if st.session_state.trip_database:
                load_name = st.selectbox("📂 讀取歷史紀錄", list(st.session_state.trip_database.keys()))
                if st.button("📖 讀取此行程"):
                    st.session_state.itinerary = copy.deepcopy(st.session_state.trip_database[load_name])
                    st.success(f"已讀取：{load_name}")
                    st.rerun()

        st.markdown("### 📍 新增行程節點")
        is_first = len(st.session_state.itinerary) == 0
        if is_first:
            st.info("👋 這是第一站！請設定出發日期與時間。")
            c1, c2 = st.columns(2)
            with c1: start_date = st.date_input("📅 起點日期")
            with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

        c1, c2, c3 = st.columns([1, 2, 1])
        with c1: city = st.text_input("🌍 城市", value="東京")
        with c2: keyword = st.text_input("🔍 找景點/機場", placeholder="例: 成田機場")
        with c3:
            st.write("")
            if st.button("🔎 搜尋", type="primary"):
                with st.spinner("雷達掃描中..."):
                    results = {}
                    try:
                        query = f"{keyword}, {city}" if city else keyword
                        locs = geolocator.geocode(query, exactly_one=False, limit=5)
                        if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                        if locs:
                            for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                    except: pass
                    st.session_state.search_results = results

        if st.session_state.search_results:
            selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))
            c1, c2, c3, c4 = st.columns(4)
            with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
            with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車', '✈️ 飛機/高鐵'], index=1)
            with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
            with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

            if st.button("➕ 確認加入行程", use_container_width=True):
                place_data = st.session_state.search_results[selected_address]
                stay_time = timedelta(minutes=stay)
                if is_first:
                    st.session_state.current_time = datetime.combine(start_date, start_time)
                    arrival_time = st.session_state.current_time
                    transport_text = "✨ 出發"
                else:
                    last_stop = st.session_state.itinerary[-1]
                    trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                    arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                    transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

                departure_time = arrival_time + stay_time
                st.session_state.itinerary.append({
                    "arrive": arrival_time.strftime("%m/%d %H:%M"),
                    "transport": transport_text, "name": place_data['name'],
                    "note": note, "stay": format_time_str(stay), "cost": cost,
                    "depart": departure_time.strftime("%m/%d %H:%M"), "coords": place_data
                })
                st.session_state.current_time = departure_time
                st.session_state.search_results = {}
                st.rerun()

        st.markdown("---")
        if st.session_state.itinerary:
            df = pd.DataFrame(st.session_state.itinerary)
            total_cost = df['cost'].sum()
            display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
            display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']
            st.dataframe(display_df, use_container_width=True, hide_index=True)
            st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總花費: NT$ {total_cost:,.0f} </h3>", unsafe_allow_html=True)

            csv = display_df.to_csv(index=False, encoding='utf-8-sig').encode('utf-8-sig')
            st.download_button(
                label="📥 下載 Excel 格式行程表 (CSV)", data=csv,
                file_name=f"Travel_Vibe_行程表.csv", mime="text/csv", type="primary"
            )

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# ==========================================
# 🔐 模組零：私有化登入系統
# ==========================================
SECRET_PASSWORD = "201020"

def check_password():
    if "password_correct" not in st.session_state: st.session_state["password_correct"] = False
    if not st.session_state["password_correct"]:
        st.markdown("""
        <div style="background: linear-gradient(135deg, #1f4037, #99f2c8); padding: 30px; border-radius: 10px; text-align: center; color: white; margin-top: 50px;">
            <h1 style="margin: 0; font-size: 36px; font-weight: bold; color: white;">🔒 Travel Vibe 私有終端</h1>
            <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">請輸入通行憑證以解鎖您的專屬行程管家</p>
        </div>
        """, unsafe_allow_html=True)
        password_input = st.text_input("🔑 通關密碼", type="password")
        if st.button("🔓 解鎖進入", use_container_width=True):
            if password_input == SECRET_PASSWORD:
                st.session_state["password_correct"] = True
                st.rerun()
            else: st.error("❌ 密碼錯誤，存取被拒絕！")
        return False
    return True

if check_password():
    # ==========================================
    # 🧠 資料庫初始化 (新增記帳資料庫)
    # ==========================================
    if 'itinerary' not in st.session_state: st.session_state.itinerary = []
    if 'trip_database' not in st.session_state: st.session_state.trip_database = {}
    if 'search_results' not in st.session_state: st.session_state.search_results = {}
    if 'current_time' not in st.session_state: st.session_state.current_time = datetime.now()

    # 🌟 新增：財務資料庫
    if 'members' not in st.session_state: st.session_state.members = []
    if 'expenses' not in st.session_state: st.session_state.expenses = []

    geolocator = Nominatim(user_agent="vibe_streamlit_app")

    visa_rules = {
        "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
        "🇭🇰 香港": "需申請『預辦登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
    }

    @st.cache_data
    def get_weather(country_name):
        try:
            loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
            if not loc: return None
            url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
            res = requests.get(url).json()
            return res['current_weather']
        except: return None

    def generate_smart_packing_list(days, country):
        packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
        if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
        else:
            packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])

        weather_data = get_weather(country)
        if weather_data:
            temp, code = weather_data['temperature'], weather_data['weathercode']
            is_raining = code >= 50
            packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
            if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
            elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
            if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])
        packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
        packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
        return packing

    def estimate_time(loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    st.markdown("""
    <div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
        <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ Travel Vibe Web App</h1>
        <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">行程 ✕ 打包 ✕ 拆帳 完美整合版 🔒</p>
    </div>
    """, unsafe_allow_html=True)

    if st.sidebar.button("🚪 登出系統"):
        st.session_state["password_correct"] = False
        st.rerun()

    # 🌟 新增第三個分頁：財務與拆帳
    tab_plan, tab_pack, tab_finance = st.tabs(['🗓️ 智慧行程', '🧳 天氣打包', '💸 記帳與拆帳'])

    # ----------------- 🧳 分頁：天氣與打包 -----------------
    with tab_pack:
        col1, col2 = st.columns(2)
        with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
        with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)
        if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
            with st.spinner(f"正在連線氣象局..."):
                result = generate_smart_packing_list(days, dest)
                for cat, items in result.items():
                    st.markdown(f"#### {cat}")
                    for item in items: st.markdown(f"- [ ] {item}")
                    st.markdown("---")

    # ----------------- 🗓️ 分頁：行程與檔案庫 -----------------
    with tab_plan:
        with st.expander("📂 歷史行程檔案管理", expanded=False):
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 東京跨年")
            with c2:
                st.write("")
                if st.button("💾 儲存"):
                    if trip_name and st.session_state.itinerary:
                        st.session_state.trip_database[trip_name] = {
                            "itinerary": copy.deepcopy(st.session_state.itinerary),
                            "members": copy.deepcopy(st.session_state.members),
                            "expenses": copy.deepcopy(st.session_state.expenses)
                        }
                        st.success(f"已儲存：{trip_name}")
            with c3:
                st.write("")
                if st.button("📄 開新"):
                    st.session_state.itinerary = []
                    st.session_state.members = []
                    st.session_state.expenses = []
                    st.rerun()
            if st.session_state.trip_database:
                load_name = st.selectbox("📂 讀取紀錄", list(st.session_state.trip_database.keys()))
                if st.button("📖 讀取此行程"):
                    db_data = st.session_state.trip_database[load_name]
                    st.session_state.itinerary = copy.deepcopy(db_data["itinerary"])
                    st.session_state.members = copy.deepcopy(db_data["members"])
                    st.session_state.expenses = copy.deepcopy(db_data["expenses"])
                    st.success(f"已讀取：{load_name}")
                    st.rerun()

        is_first = len(st.session_state.itinerary) == 0
        if is_first:
            c1, c2 = st.columns(2)
            with c1: start_date = st.date_input("📅 起點日期")
            with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

        c1, c2, c3 = st.columns([1, 2, 1])
        with c1: city = st.text_input("🌍 城市", value="東京")
        with c2: keyword = st.text_input("🔍 找景點", placeholder="例: 成田機場")
        with c3:
            st.write("")
            if st.button("🔎 搜尋", type="primary"):
                with st.spinner("掃描中..."):
                    results = {}
                    try:
                        query = f"{keyword}, {city}" if city else keyword
                        locs = geolocator.geocode(query, exactly_one=False, limit=5)
                        if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                        if locs:
                            for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                    except: pass
                    st.session_state.search_results = results

        if st.session_state.search_results:
            selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))
            c1, c2, c3, c4 = st.columns(4)
            with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
            with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車'], index=1)
            with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
            with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

            if st.button("➕ 確認加入", use_container_width=True):
                place_data = st.session_state.search_results[selected_address]
                stay_time = timedelta(minutes=stay)
                if is_first:
                    st.session_state.current_time = datetime.combine(start_date, start_time)
                    arrival_time = st.session_state.current_time
                    transport_text = "✨ 出發"
                else:
                    last_stop = st.session_state.itinerary[-1]
                    trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                    arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                    transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

                departure_time = arrival_time + stay_time
                st.session_state.itinerary.append({
                    "arrive": arrival_time.strftime("%m/%d %H:%M"), "transport": transport_text,
                    "name": place_data['name'], "note": note, "stay": format_time_str(stay),
                    "cost": cost, "depart": departure_time.strftime("%m/%d %H:%M"), "coords": place_data
                })
                st.session_state.current_time = departure_time
                st.session_state.search_results = {}
                st.rerun()

        st.markdown("---")
        if st.session_state.itinerary:
            df = pd.DataFrame(st.session_state.itinerary)
            display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
            display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']
            st.dataframe(display_df, use_container_width=True, hide_index=True)
            st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總行程預算: NT$ {df['cost'].sum():,.0f} </h3>", unsafe_allow_html=True)

    # ----------------- 💸 分頁：記帳與拆帳 -----------------
    with tab_finance:
        st.markdown("### 👥 1. 設定旅伴")
        members_input = st.text_input("請輸入旅伴名字 (用半形逗號 , 分開)", value="小明, 小華, 媽媽")
        if st.button("💾 更新旅伴名單"):
            # 將字串轉換為乾淨的列表
            st.session_state.members = [m.strip() for m in members_input.split(",") if m.strip()]
            st.success(f"已更新旅伴：{', '.join(st.session_state.members)}")

        st.markdown("---")

        st.markdown("### ✍️ 2. 紀錄花費 (記帳)")
        if not st.session_state.members:
            st.warning("⚠️ 請先在上方輸入旅伴名字！")
        else:
            c1, c2, c3 = st.columns(3)
            with c1: exp_item = st.text_input("消費項目", placeholder="例: 晚餐燒肉")
            with c2: exp_amount = st.number_input("金額 (NT$)", min_value=0, step=100)
            with c3: exp_payer = st.selectbox("誰先代墊的？", st.session_state.members)

            if st.button("➕ 新增這筆帳款", type="primary"):
                if exp_item and exp_amount > 0:
                    st.session_state.expenses.append({
                        "項目": exp_item,
                        "金額": exp_amount,
                        "代墊人": exp_payer,
                        "分攤方式": "大家平分" # 目前 MVP 先做最實用的均分
                    })
                    st.success("✅ 記帳成功！")
                else:
                    st.error("請輸入項目和金額")

        # 顯示記帳明細
        if st.session_state.expenses:
            st.markdown("#### 📋 帳單明細")
            exp_df = pd.DataFrame(st.session_state.expenses)
            st.dataframe(exp_df, use_container_width=True, hide_index=True)

            if st.button("🗑️ 清空所有帳單", type="secondary"):
                st.session_state.expenses = []
                st.rerun()

            st.markdown("---")

            # 🚀 核心：AI 拆帳演算法
            st.markdown("### 🤖 3. 一鍵智慧結算 (誰該給誰多少錢)")

            # 計算每個人總共付了多少，以及應該付多少
            balances = {member: 0 for member in st.session_state.members}
            total_expense = 0

            for exp in st.session_state.expenses:
                amount = exp["金額"]
                payer = exp["代墊人"]
                total_expense += amount

                # 代墊人多出了錢 (餘額增加)
                if payer in balances: balances[payer] += amount

                # 每個人都要平分這筆錢 (餘額減少)
                split_amount = amount / len(st.session_state.members)
                for m in st.session_state.members:
                    if m in balances: balances[m] -= split_amount

            st.info(f"💵 團隊總花費：NT$ {total_expense:,.0f} (每人應平均負擔：NT$ {total_expense/len(st.session_state.members):,.0f})")

            # 結算顯示
            st.markdown("#### 💰 結算結果：")
            for member, balance in balances.items():
                if balance > 1: # 大於 0 代表他代墊比較多，別人要給他錢
                    st.success(f"**{member}** 可以拿回：NT$ {balance:,.0f}")
                elif balance < -1: # 小於 0 代表他付比較少，要掏錢出來
                    st.error(f"**{member}** 需要支付：NT$ {abs(balance):,.0f}")
                else:
                    st.write(f"**{member}** 不欠錢，完美打平！")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# ==========================================
# 🔐 模組零：私有化登入系統
# ==========================================
SECRET_PASSWORD = "201020"

def check_password():
    if "password_correct" not in st.session_state: st.session_state["password_correct"] = False
    if not st.session_state["password_correct"]:
        st.markdown("""
        <div style="background: linear-gradient(135deg, #1f4037, #99f2c8); padding: 30px; border-radius: 10px; text-align: center; color: white; margin-top: 50px;">
            <h1 style="margin: 0; font-size: 36px; font-weight: bold; color: white;">🔒 Travel Vibe 私有終端</h1>
            <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">請輸入通行憑證以解鎖您的專屬行程管家</p>
        </div>
        """, unsafe_allow_html=True)
        password_input = st.text_input("🔑 通關密碼", type="password")
        if st.button("🔓 解鎖進入", use_container_width=True):
            if password_input == SECRET_PASSWORD:
                st.session_state["password_correct"] = True
                st.rerun()
            else: st.error("❌ 密碼錯誤，存取被拒絕！")
        return False
    return True

if check_password():
    # ==========================================
    # 🧠 資料庫與 API 引擎
    # ==========================================
    if 'itinerary' not in st.session_state: st.session_state.itinerary = []
    if 'trip_database' not in st.session_state: st.session_state.trip_database = {}
    if 'search_results' not in st.session_state: st.session_state.search_results = {}
    if 'current_time' not in st.session_state: st.session_state.current_time = datetime.now()
    if 'members' not in st.session_state: st.session_state.members = []
    if 'expenses' not in st.session_state: st.session_state.expenses = []

    geolocator = Nominatim(user_agent="vibe_streamlit_app")

    visa_rules = {
        "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
        "🇭🇰 香港": "需申請『預辦登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
    }

    @st.cache_data(ttl=3600) # 快取 1 小時，避免頻繁呼叫
    def get_weather(country_name):
        try:
            loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
            if not loc: return None
            url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
            res = requests.get(url).json()
            return res['current_weather']
        except: return None

    # 🌟 新增：即時匯率 API (以台幣為基準)
    @st.cache_data(ttl=3600)
    def get_exchange_rates():
        try:
            url = "https://api.exchangerate-api.com/v4/latest/TWD"
            res = requests.get(url).json()
            return res.get("rates", {})
        except:
            # 萬一 API 壞掉的備用預設匯率 (大約值)
            return {"TWD": 1.0, "JPY": 4.7, "KRW": 42.0, "USD": 0.031, "EUR": 0.029, "HKD": 0.24, "CNY": 0.22}

    def generate_smart_packing_list(days, country):
        packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
        if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
        else:
            packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])

        weather_data = get_weather(country)
        if weather_data:
            temp, code = weather_data['temperature'], weather_data['weathercode']
            is_raining = code >= 50
            packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
            if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
            elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
            if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])
        packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
        packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
        return packing

    def estimate_time(loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    st.markdown("""
    <div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
        <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ Travel Vibe Web App</h1>
        <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">行程 ✕ 打包 ✕ 即時匯率拆帳 🔒</p>
    </div>
    """, unsafe_allow_html=True)

    if st.sidebar.button("🚪 登出系統"):
        st.session_state["password_correct"] = False
        st.rerun()

    tab_plan, tab_pack, tab_finance = st.tabs(['🗓️ 智慧行程', '🧳 天氣打包', '💸 匯率與拆帳'])

    # ----------------- 🧳 分頁：天氣與打包 -----------------
    with tab_pack:
        col1, col2 = st.columns(2)
        with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
        with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)
        if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
            with st.spinner(f"正在連線氣象局..."):
                result = generate_smart_packing_list(days, dest)
                for cat, items in result.items():
                    st.markdown(f"#### {cat}")
                    for item in items: st.markdown(f"- [ ] {item}")
                    st.markdown("---")

    # ----------------- 🗓️ 分頁：行程與檔案庫 -----------------
    with tab_plan:
        with st.expander("📂 歷史行程檔案管理", expanded=False):
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 東京跨年")
            with c2:
                st.write("")
                if st.button("💾 儲存"):
                    if trip_name and st.session_state.itinerary:
                        st.session_state.trip_database[trip_name] = {
                            "itinerary": copy.deepcopy(st.session_state.itinerary),
                            "members": copy.deepcopy(st.session_state.members),
                            "expenses": copy.deepcopy(st.session_state.expenses)
                        }
                        st.success(f"已儲存：{trip_name}")
            with c3:
                st.write("")
                if st.button("📄 開新"):
                    st.session_state.itinerary = []
                    st.session_state.members = []
                    st.session_state.expenses = []
                    st.rerun()
            if st.session_state.trip_database:
                load_name = st.selectbox("📂 讀取紀錄", list(st.session_state.trip_database.keys()))
                if st.button("📖 讀取此行程"):
                    db_data = st.session_state.trip_database[load_name]
                    st.session_state.itinerary = copy.deepcopy(db_data["itinerary"])
                    st.session_state.members = copy.deepcopy(db_data["members"])
                    st.session_state.expenses = copy.deepcopy(db_data["expenses"])
                    st.success(f"已讀取：{load_name}")
                    st.rerun()

        is_first = len(st.session_state.itinerary) == 0
        if is_first:
            c1, c2 = st.columns(2)
            with c1: start_date = st.date_input("📅 起點日期")
            with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

        c1, c2, c3 = st.columns([1, 2, 1])
        with c1: city = st.text_input("🌍 城市", value="東京")
        with c2: keyword = st.text_input("🔍 找景點", placeholder="例: 成田機場")
        with c3:
            st.write("")
            if st.button("🔎 搜尋", type="primary"):
                with st.spinner("掃描中..."):
                    results = {}
                    try:
                        query = f"{keyword}, {city}" if city else keyword
                        locs = geolocator.geocode(query, exactly_one=False, limit=5)
                        if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                        if locs:
                            for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                    except: pass
                    st.session_state.search_results = results

        if st.session_state.search_results:
            selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))
            c1, c2, c3, c4 = st.columns(4)
            with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
            with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車'], index=1)
            with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
            with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

            if st.button("➕ 確認加入", use_container_width=True):
                place_data = st.session_state.search_results[selected_address]
                stay_time = timedelta(minutes=stay)
                if is_first:
                    st.session_state.current_time = datetime.combine(start_date, start_time)
                    arrival_time = st.session_state.current_time
                    transport_text = "✨ 出發"
                else:
                    last_stop = st.session_state.itinerary[-1]
                    trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                    arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                    transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

                departure_time = arrival_time + stay_time
                st.session_state.itinerary.append({
                    "arrive": arrival_time.strftime("%m/%d %H:%M"), "transport": transport_text,
                    "name": place_data['name'], "note": note, "stay": format_time_str(stay),
                    "cost": cost, "depart": departure_time.strftime("%m/%d %H:%M"), "coords": place_data
                })
                st.session_state.current_time = departure_time
                st.session_state.search_results = {}
                st.rerun()

        st.markdown("---")
        if st.session_state.itinerary:
            df = pd.DataFrame(st.session_state.itinerary)
            display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
            display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']
            st.dataframe(display_df, use_container_width=True, hide_index=True)
            st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總行程預算: NT$ {df['cost'].sum():,.0f} </h3>", unsafe_allow_html=True)

    # ----------------- 💸 分頁：記帳與即時匯率拆帳 -----------------
    with tab_finance:
        st.markdown("### 👥 1. 設定旅伴")
        members_input = st.text_input("請輸入旅伴名字 (用半形逗號 , 分開)", value="小明, 小華, 媽媽")
        if st.button("💾 更新旅伴名單"):
            st.session_state.members = [m.strip() for m in members_input.split(",") if m.strip()]
            st.success(f"已更新旅伴：{', '.join(st.session_state.members)}")

        st.markdown("---")

        # 🌟 匯率換算與記帳區塊
        st.markdown("### 💱 2. 跨國記帳 (自動轉台幣)")
        if not st.session_state.members:
            st.warning("⚠️ 請先在上方輸入旅伴名字！")
        else:
            # 取得即時匯率
            rates = get_exchange_rates()

            c1, c2, c3, c4 = st.columns([2, 1.5, 1.5, 1.5])
            with c1: exp_item = st.text_input("消費項目", placeholder="例: 敘敘苑燒肉")
            with c2:
                # 幣別選單
                currency_options = {"🇹🇼 台幣 (TWD)": "TWD", "🇯🇵 日幣 (JPY)": "JPY", "🇰🇷 韓元 (KRW)": "KRW", "🇺🇸 美金 (USD)": "USD", "🇪🇺 歐元 (EUR)": "EUR", "🇭🇰 港幣 (HKD)": "HKD", "🇨🇳 人民幣 (CNY)": "CNY"}
                curr_label = st.selectbox("幣別", list(currency_options.keys()))
                curr_code = currency_options[curr_label]
            with c3: exp_amount = st.number_input(f"外幣金額", min_value=0.0, step=100.0)
            with c4: exp_payer = st.selectbox("誰先代墊？", st.session_state.members)

            # 即時預覽折合台幣多少錢
            if curr_code in rates and rates[curr_code] > 0:
                # 算法：台幣基準下，當地貨幣轉台幣 = 外幣金額 / 該外幣對台幣的匯率
                twd_estimate = exp_amount / rates[curr_code]
            else:
                twd_estimate = exp_amount # 萬一抓不到就一比一

            st.caption(f"💡 即時匯率換算：約折合 **NT$ {twd_estimate:,.0f}**")

            if st.button("➕ 新增這筆帳款", type="primary"):
                if exp_item and exp_amount > 0:
                    st.session_state.expenses.append({
                        "項目": exp_item,
                        "原幣別": curr_code,
                        "原幣金額": exp_amount,
                        "折合台幣 (NT$)": round(twd_estimate), # 直接存入換算後的台幣
                        "代墊人": exp_payer
                    })
                    st.success(f"✅ 成功記帳！已自動轉換為台幣 NT$ {round(twd_estimate)}")
                    st.rerun()
                else:
                    st.error("請輸入項目和金額")

        if st.session_state.expenses:
            st.markdown("#### 📋 帳單明細")
            exp_df = pd.DataFrame(st.session_state.expenses)
            st.dataframe(exp_df, use_container_width=True, hide_index=True)

            if st.button("🗑️ 清空所有帳單", type="secondary"):
                st.session_state.expenses = []
                st.rerun()

            st.markdown("---")

            st.markdown("### 🤖 3. 一鍵智慧結算 (統一以台幣結算)")
            balances = {member: 0 for member in st.session_state.members}
            total_expense_twd = 0

            for exp in st.session_state.expenses:
                amount_twd = exp["折合台幣 (NT$)"]
                payer = exp["代墊人"]
                total_expense_twd += amount_twd

                if payer in balances: balances[payer] += amount_twd
                split_amount = amount_twd / len(st.session_state.members)
                for m in st.session_state.members:
                    if m in balances: balances[m] -= split_amount

            st.info(f"💵 團隊總花費：NT$ {total_expense_twd:,.0f} (每人應平均負擔：NT$ {total_expense_twd/len(st.session_state.members):,.0f})")

            st.markdown("#### 💰 最終轉帳指示：")
            for member, balance in balances.items():
                if balance > 1:
                    st.success(f"收款方 🟢 **{member}** 可以拿回：NT$ {balance:,.0f}")
                elif balance < -1:
                    st.error(f"付款方 🔴 **{member}** 需要支付：NT$ {abs(balance):,.0f}")
                else:
                    st.write(f"平手 ⚪ **{member}** 不欠錢，完美打平！")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# ==========================================
# 🔐 模組零：私有化登入系統
# ==========================================
SECRET_PASSWORD = "201020"

def check_password():
    if "password_correct" not in st.session_state: st.session_state["password_correct"] = False
    if not st.session_state["password_correct"]:
        st.markdown("""
        <div style="background: linear-gradient(135deg, #1f4037, #99f2c8); padding: 30px; border-radius: 10px; text-align: center; color: white; margin-top: 50px;">
            <h1 style="margin: 0; font-size: 36px; font-weight: bold; color: white;">🔒 Travel Vibe 私有終端</h1>
            <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">請輸入通行憑證以解鎖您的專屬行程管家</p>
        </div>
        """, unsafe_allow_html=True)
        password_input = st.text_input("🔑 通關密碼", type="password")
        if st.button("🔓 解鎖進入", use_container_width=True):
            if password_input == SECRET_PASSWORD:
                st.session_state["password_correct"] = True
                st.rerun()
            else: st.error("❌ 密碼錯誤，存取被拒絕！")
        return False
    return True

if check_password():
    # ==========================================
    # 🧠 資料庫與 API 引擎
    # ==========================================
    if 'itinerary' not in st.session_state: st.session_state.itinerary = []
    if 'trip_database' not in st.session_state: st.session_state.trip_database = {}
    if 'search_results' not in st.session_state: st.session_state.search_results = {}
    if 'current_time' not in st.session_state: st.session_state.current_time = datetime.now()
    if 'members' not in st.session_state: st.session_state.members = []
    if 'expenses' not in st.session_state: st.session_state.expenses = []

    geolocator = Nominatim(user_agent="vibe_streamlit_app")
    visa_rules = {
        "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
        "🇭🇰 香港": "需申請『預辦登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
    }

    @st.cache_data(ttl=3600)
    def get_weather(country_name):
        try:
            loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
            if not loc: return None
            url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
            res = requests.get(url).json()
            return res['current_weather']
        except: return None

    @st.cache_data(ttl=3600)
    def get_exchange_rates():
        try:
            url = "https://api.exchangerate-api.com/v4/latest/TWD"
            res = requests.get(url).json()
            return res.get("rates", {})
        except:
            return {"TWD": 1.0, "JPY": 4.7, "KRW": 42.0, "USD": 0.031, "EUR": 0.029, "HKD": 0.24, "CNY": 0.22}

    def generate_smart_packing_list(days, country):
        packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
        if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
        else:
            packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])

        weather_data = get_weather(country)
        if weather_data:
            temp, code = weather_data['temperature'], weather_data['weathercode']
            is_raining = code >= 50
            packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
            if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
            elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
            if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])
        packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
        packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
        return packing

    def estimate_time(loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    st.markdown("""
    <div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
        <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ Travel Vibe Web App</h1>
        <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">彈性拆帳 ✕ 即時匯率 ✕ 完美規劃 🔒</p>
    </div>
    """, unsafe_allow_html=True)

    if st.sidebar.button("🚪 登出系統"):
        st.session_state["password_correct"] = False
        st.rerun()

    tab_plan, tab_pack, tab_finance = st.tabs(['🗓️ 智慧行程', '🧳 天氣打包', '💸 匯率與拆帳'])

    # ----------------- 🧳 分頁：天氣與打包 -----------------
    with tab_pack:
        col1, col2 = st.columns(2)
        with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
        with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)
        if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
            with st.spinner(f"正在連線氣象局..."):
                result = generate_smart_packing_list(days, dest)
                for cat, items in result.items():
                    st.markdown(f"#### {cat}")
                    for item in items: st.markdown(f"- [ ] {item}")
                    st.markdown("---")

    # ----------------- 🗓️ 分頁：行程與檔案庫 -----------------
    with tab_plan:
        with st.expander("📂 歷史行程檔案管理", expanded=False):
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 東京跨年")
            with c2:
                st.write("")
                if st.button("💾 儲存"):
                    if trip_name and st.session_state.itinerary:
                        st.session_state.trip_database[trip_name] = {
                            "itinerary": copy.deepcopy(st.session_state.itinerary),
                            "members": copy.deepcopy(st.session_state.members),
                            "expenses": copy.deepcopy(st.session_state.expenses)
                        }
                        st.success(f"已儲存：{trip_name}")
            with c3:
                st.write("")
                if st.button("📄 開新"):
                    st.session_state.itinerary = []
                    st.session_state.members = []
                    st.session_state.expenses = []
                    st.rerun()
            if st.session_state.trip_database:
                load_name = st.selectbox("📂 讀取紀錄", list(st.session_state.trip_database.keys()))
                if st.button("📖 讀取此行程"):
                    db_data = st.session_state.trip_database[load_name]
                    st.session_state.itinerary = copy.deepcopy(db_data["itinerary"])
                    st.session_state.members = copy.deepcopy(db_data["members"])
                    st.session_state.expenses = copy.deepcopy(db_data["expenses"])
                    st.success(f"已讀取：{load_name}")
                    st.rerun()

        is_first = len(st.session_state.itinerary) == 0
        if is_first:
            c1, c2 = st.columns(2)
            with c1: start_date = st.date_input("📅 起點日期")
            with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

        c1, c2, c3 = st.columns([1, 2, 1])
        with c1: city = st.text_input("🌍 城市", value="東京")
        with c2: keyword = st.text_input("🔍 找景點", placeholder="例: 成田機場")
        with c3:
            st.write("")
            if st.button("🔎 搜尋", type="primary"):
                with st.spinner("掃描中..."):
                    results = {}
                    try:
                        query = f"{keyword}, {city}" if city else keyword
                        locs = geolocator.geocode(query, exactly_one=False, limit=5)
                        if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                        if locs:
                            for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                    except: pass
                    st.session_state.search_results = results

        if st.session_state.search_results:
            selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))
            c1, c2, c3, c4 = st.columns(4)
            with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
            with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車'], index=1)
            with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
            with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

            if st.button("➕ 確認加入", use_container_width=True):
                place_data = st.session_state.search_results[selected_address]
                stay_time = timedelta(minutes=stay)
                if is_first:
                    st.session_state.current_time = datetime.combine(start_date, start_time)
                    arrival_time = st.session_state.current_time
                    transport_text = "✨ 出發"
                else:
                    last_stop = st.session_state.itinerary[-1]
                    trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                    arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                    transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

                departure_time = arrival_time + stay_time
                st.session_state.itinerary.append({
                    "arrive": arrival_time.strftime("%m/%d %H:%M"), "transport": transport_text,
                    "name": place_data['name'], "note": note, "stay": format_time_str(stay),
                    "cost": cost, "depart": departure_time.strftime("%m/%d %H:%M"), "coords": place_data
                })
                st.session_state.current_time = departure_time
                st.session_state.search_results = {}
                st.rerun()

        st.markdown("---")
        if st.session_state.itinerary:
            df = pd.DataFrame(st.session_state.itinerary)
            display_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
            display_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']
            st.dataframe(display_df, use_container_width=True, hide_index=True)
            st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總行程預算: NT$ {df['cost'].sum():,.0f} </h3>", unsafe_allow_html=True)

    # ----------------- 💸 分頁：記帳與動態彈性拆帳 -----------------
    with tab_finance:
        st.markdown("### 👥 1. 設定旅伴")
        members_input = st.text_input("請輸入旅伴名字 (用半形逗號 , 分開)", value="Alice, Bob, Kevin")
        if st.button("💾 更新旅伴名單"):
            st.session_state.members = [m.strip() for m in members_input.split(",") if m.strip()]
            st.success(f"已更新旅伴：{', '.join(st.session_state.members)}")

        st.markdown("---")

        # 🌟 進階動態記帳區塊
        st.markdown("### 💱 2. 跨國記帳 (動態分攤版)")
        if not st.session_state.members:
            st.warning("⚠️ 請先在上方輸入旅伴名字！")
        else:
            rates = get_exchange_rates()

            # 第一排：消費項目與金額
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: exp_item = st.text_input("消費項目", placeholder="例: 敘敘苑燒肉、和服體驗")
            with c2:
                currency_options = {"🇹🇼 台幣 (TWD)": "TWD", "🇯🇵 日幣 (JPY)": "JPY", "🇰🇷 韓元 (KRW)": "KRW", "🇺🇸 美金 (USD)": "USD"}
                curr_label = st.selectbox("幣別", list(currency_options.keys()))
                curr_code = currency_options[curr_label]
            with c3: exp_amount = st.number_input(f"外幣金額", min_value=0.0, step=100.0)

            # 第二排：動態多選拆帳邏輯 🌟
            c4, c5 = st.columns(2)
            with c4:
                # 多選：可以兩個人一起付
                exp_payers = st.multiselect("💳 誰付的錢？(可多選)", st.session_state.members, default=st.session_state.members[0:1] if st.session_state.members else [])
            with c5:
                # 多選：誰參與了這次消費
                exp_consumers = st.multiselect("🍽️ 分攤給誰？(打叉可排除未參與者)", st.session_state.members, default=st.session_state.members)

            if curr_code in rates and rates[curr_code] > 0:
                twd_estimate = exp_amount / rates[curr_code]
            else: twd_estimate = exp_amount

            st.caption(f"💡 即時匯率換算：約折合 **NT$ {twd_estimate:,.0f}**")

            if st.button("➕ 新增這筆帳款", type="primary"):
                if not exp_item or exp_amount <= 0:
                    st.error("❌ 請輸入項目名稱與大於 0 的金額！")
                elif not exp_payers:
                    st.error("❌ 至少要選一個付款人！")
                elif not exp_consumers:
                    st.error("❌ 至少要有一個人來分攤這筆費用！")
                else:
                    st.session_state.expenses.append({
                        "項目": exp_item,
                        "原幣別": curr_code,
                        "折合台幣": round(twd_estimate),
                        "付款人": ", ".join(exp_payers),
                        "分攤給": ", ".join(exp_consumers),
                        "_payers_list": exp_payers,         # 隱藏欄位供系統結算用
                        "_consumers_list": exp_consumers    # 隱藏欄位供系統結算用
                    })
                    st.success(f"✅ 記帳成功！")
                    st.rerun()

        if st.session_state.expenses:
            st.markdown("#### 📋 帳單明細")
            exp_df = pd.DataFrame(st.session_state.expenses)

            # 只顯示給使用者看乾淨的欄位
            display_exp_df = exp_df[["項目", "原幣別", "折合台幣", "付款人", "分攤給"]]
            st.dataframe(display_df, use_container_width=True, hide_index=True)

            if st.button("🗑️ 清空所有帳單", type="secondary"):
                st.session_state.expenses = []
                st.rerun()

            st.markdown("---")

            # 🚀 核心：動態拆帳 AI 演算法
            st.markdown("### 🤖 3. 智慧動態結算 (TWD)")
            balances = {member: 0.0 for member in st.session_state.members}
            total_expense_twd = 0.0

            for exp in st.session_state.expenses:
                amount_twd = exp["折合台幣"]
                payers = exp["_payers_list"]
                consumers = exp["_consumers_list"]
                total_expense_twd += amount_twd

                # 1. 付款人增加餘額 (如果是兩個人合付，假設他們各出一半)
                payer_split = amount_twd / len(payers)
                for p in payers:
                    if p in balances: balances[p] += payer_split

                # 2. 分攤人扣除餘額
                consumer_split = amount_twd / len(consumers)
                for c in consumers:
                    if c in balances: balances[c] -= consumer_split

            st.info(f"💵 團隊總花費：NT$ {total_expense_twd:,.0f}")

            st.markdown("#### 💰 最終轉帳指示：")
            for member, balance in balances.items():
                if balance > 1:
                    st.success(f"收款方 🟢 **{member}** 可以拿回：NT$ {balance:,.0f}")
                elif balance < -1:
                    st.error(f"付款方 🔴 **{member}** 需要支付：NT$ {abs(balance):,.0f}")
                else:
                    st.write(f"平手 ⚪ **{member}** 不欠錢，完美打平！")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# ==========================================
# 🔐 模組零：私有化登入系統
# ==========================================
SECRET_PASSWORD = "201020"

def check_password():
    if "password_correct" not in st.session_state: st.session_state["password_correct"] = False
    if not st.session_state["password_correct"]:
        st.markdown("""
        <div style="background: linear-gradient(135deg, #1f4037, #99f2c8); padding: 30px; border-radius: 10px; text-align: center; color: white; margin-top: 50px;">
            <h1 style="margin: 0; font-size: 36px; font-weight: bold; color: white;">🔒 Travel Vibe 私有終端</h1>
            <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">請輸入通行憑證以解鎖您的專屬行程管家</p>
        </div>
        """, unsafe_allow_html=True)
        password_input = st.text_input("🔑 通關密碼", type="password")
        if st.button("🔓 解鎖進入", use_container_width=True):
            if password_input == SECRET_PASSWORD:
                st.session_state["password_correct"] = True
                st.rerun()
            else: st.error("❌ 密碼錯誤，存取被拒絕！")
        return False
    return True

if check_password():
    # ==========================================
    # 🧠 資料庫與 API 引擎
    # ==========================================
    if 'itinerary' not in st.session_state: st.session_state.itinerary = []
    if 'trip_database' not in st.session_state: st.session_state.trip_database = {}
    if 'search_results' not in st.session_state: st.session_state.search_results = {}
    if 'current_time' not in st.session_state: st.session_state.current_time = datetime.now()
    if 'members' not in st.session_state: st.session_state.members = []
    if 'expenses' not in st.session_state: st.session_state.expenses = []

    geolocator = Nominatim(user_agent="vibe_streamlit_app")
    visa_rules = {
        "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
        "🇭🇰 香港": "需申請『預辦登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
    }

    @st.cache_data(ttl=3600)
    def get_weather(country_name):
        try:
            loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
            if not loc: return None
            url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
            res = requests.get(url).json()
            return res['current_weather']
        except: return None

    @st.cache_data(ttl=3600)
    def get_exchange_rates():
        try:
            url = "https://api.exchangerate-api.com/v4/latest/TWD"
            res = requests.get(url).json()
            return res.get("rates", {})
        except:
            return {"TWD": 1.0, "JPY": 4.7, "KRW": 42.0, "USD": 0.031, "EUR": 0.029, "HKD": 0.24, "CNY": 0.22}

    def generate_smart_packing_list(days, country):
        packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
        if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
        else:
            packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])

        weather_data = get_weather(country)
        if weather_data:
            temp, code = weather_data['temperature'], weather_data['weathercode']
            is_raining = code >= 50
            packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
            if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
            elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
            if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])
        packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
        packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
        return packing

    def estimate_time(loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    st.markdown("""
    <div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
        <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ Travel Vibe Web App</h1>
        <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">彈性拆帳 ✕ 即時匯率 ✕ 完美規劃 🔒</p>
    </div>
    """, unsafe_allow_html=True)

    if st.sidebar.button("🚪 登出系統"):
        st.session_state["password_correct"] = False
        st.rerun()

    tab_plan, tab_pack, tab_finance = st.tabs(['🗓️ 智慧行程', '🧳 天氣打包', '💸 匯率與拆帳'])

    # ----------------- 🧳 分頁：天氣與打包 -----------------
    with tab_pack:
        col1, col2 = st.columns(2)
        with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
        with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)
        if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
            with st.spinner(f"正在連線氣象局..."):
                result = generate_smart_packing_list(days, dest)
                for cat, items in result.items():
                    st.markdown(f"#### {cat}")
                    for item in items: st.markdown(f"- [ ] {item}")
                    st.markdown("---")

    # ----------------- 🗓️ 分頁：行程與檔案庫 -----------------
    with tab_plan:
        with st.expander("📂 歷史行程檔案管理", expanded=False):
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 東京跨年")
            with c2:
                st.write("")
                if st.button("💾 儲存"):
                    if trip_name and st.session_state.itinerary:
                        st.session_state.trip_database[trip_name] = {
                            "itinerary": copy.deepcopy(st.session_state.itinerary),
                            "members": copy.deepcopy(st.session_state.members),
                            "expenses": copy.deepcopy(st.session_state.expenses)
                        }
                        st.success(f"已儲存：{trip_name}")
            with c3:
                st.write("")
                if st.button("📄 開新"):
                    st.session_state.itinerary = []
                    st.session_state.members = []
                    st.session_state.expenses = []
                    st.rerun()
            if st.session_state.trip_database:
                load_name = st.selectbox("📂 讀取紀錄", list(st.session_state.trip_database.keys()))
                if st.button("📖 讀取此行程"):
                    db_data = st.session_state.trip_database[load_name]
                    st.session_state.itinerary = copy.deepcopy(db_data["itinerary"])
                    st.session_state.members = copy.deepcopy(db_data["members"])
                    st.session_state.expenses = copy.deepcopy(db_data["expenses"])
                    st.success(f"已讀取：{load_name}")
                    st.rerun()

        is_first = len(st.session_state.itinerary) == 0
        if is_first:
            c1, c2 = st.columns(2)
            with c1: start_date = st.date_input("📅 起點日期")
            with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

        c1, c2, c3 = st.columns([1, 2, 1])
        with c1: city = st.text_input("🌍 城市", value="東京")
        with c2: keyword = st.text_input("🔍 找景點", placeholder="例: 成田機場")
        with c3:
            st.write("")
            if st.button("🔎 搜尋", type="primary"):
                with st.spinner("掃描中..."):
                    results = {}
                    try:
                        query = f"{keyword}, {city}" if city else keyword
                        locs = geolocator.geocode(query, exactly_one=False, limit=5)
                        if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                        if locs:
                            for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                    except: pass
                    st.session_state.search_results = results

        if st.session_state.search_results:
            selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))
            c1, c2, c3, c4 = st.columns(4)
            with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
            with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車'], index=1)
            with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
            with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

            if st.button("➕ 確認加入", use_container_width=True):
                place_data = st.session_state.search_results[selected_address]
                stay_time = timedelta(minutes=stay)
                if is_first:
                    st.session_state.current_time = datetime.combine(start_date, start_time)
                    arrival_time = st.session_state.current_time
                    transport_text = "✨ 出發"
                else:
                    last_stop = st.session_state.itinerary[-1]
                    trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                    arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                    transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

                departure_time = arrival_time + stay_time
                st.session_state.itinerary.append({
                    "arrive": arrival_time.strftime("%m/%d %H:%M"), "transport": transport_text,
                    "name": place_data['name'], "note": note, "stay": format_time_str(stay),
                    "cost": cost, "depart": departure_time.strftime("%m/%d %H:%M"), "coords": place_data
                })
                st.session_state.current_time = departure_time
                st.session_state.search_results = {}
                st.rerun()

        st.markdown("---")
        if st.session_state.itinerary:
            df = pd.DataFrame(st.session_state.itinerary)
            # 行程表的 DataFrame 叫做 display_plan_df
            display_plan_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
            display_plan_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']
            st.dataframe(display_plan_df, use_container_width=True, hide_index=True)
            st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總行程預算: NT$ {df['cost'].sum():,.0f} </h3>", unsafe_allow_html=True)

    # ----------------- 💸 分頁：記帳與動態彈性拆帳 -----------------
    with tab_finance:
        st.markdown("### 👥 1. 設定旅伴")
        members_input = st.text_input("請輸入旅伴名字 (用半形逗號 , 分開)", value="Alice, Bob, Kevin")
        if st.button("💾 更新旅伴名單"):
            st.session_state.members = [m.strip() for m in members_input.split(",") if m.strip()]
            st.success(f"已更新旅伴：{', '.join(st.session_state.members)}")

        st.markdown("---")

        st.markdown("### 💱 2. 跨國記帳 (動態分攤版)")
        if not st.session_state.members:
            st.warning("⚠️ 請先在上方輸入旅伴名字！")
        else:
            rates = get_exchange_rates()

            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: exp_item = st.text_input("消費項目", placeholder="例: 敘敘苑燒肉、和服體驗")
            with c2:
                currency_options = {"🇹🇼 台幣 (TWD)": "TWD", "🇯🇵 日幣 (JPY)": "JPY", "🇰🇷 韓元 (KRW)": "KRW", "🇺🇸 美金 (USD)": "USD"}
                curr_label = st.selectbox("幣別", list(currency_options.keys()))
                curr_code = currency_options[curr_label]
            with c3: exp_amount = st.number_input(f"外幣金額", min_value=0.0, step=100.0)

            c4, c5 = st.columns(2)
            with c4:
                exp_payers = st.multiselect("💳 誰付的錢？(可多選)", st.session_state.members, default=st.session_state.members[0:1] if st.session_state.members else [])
            with c5:
                exp_consumers = st.multiselect("🍽️ 分攤給誰？(打叉可排除未參與者)", st.session_state.members, default=st.session_state.members)

            if curr_code in rates and rates[curr_code] > 0:
                twd_estimate = exp_amount / rates[curr_code]
            else: twd_estimate = exp_amount

            st.caption(f"💡 即時匯率換算：約折合 **NT$ {twd_estimate:,.0f}**")

            if st.button("➕ 新增這筆帳款", type="primary"):
                if not exp_item or exp_amount <= 0:
                    st.error("❌ 請輸入項目名稱與大於 0 的金額！")
                elif not exp_payers:
                    st.error("❌ 至少要選一個付款人！")
                elif not exp_consumers:
                    st.error("❌ 至少要有一個人來分攤這筆費用！")
                else:
                    st.session_state.expenses.append({
                        "項目": exp_item,
                        "原幣別": curr_code,
                        "折合台幣": round(twd_estimate),
                        "付款人": ", ".join(exp_payers),
                        "分攤給": ", ".join(exp_consumers),
                        "_payers_list": exp_payers,
                        "_consumers_list": exp_consumers
                    })
                    st.success(f"✅ 記帳成功！")
                    st.rerun()

        if st.session_state.expenses:
            st.markdown("#### 📋 帳單明細")
            exp_df = pd.DataFrame(st.session_state.expenses)

            # 🐞 修復：這裡的變數名稱已經修正為 display_exp_df，絕不會再報錯！
            display_exp_df = exp_df[["項目", "原幣別", "折合台幣", "付款人", "分攤給"]]
            st.dataframe(display_exp_df, use_container_width=True, hide_index=True)

            if st.button("🗑️ 清空所有帳單", type="secondary"):
                st.session_state.expenses = []
                st.rerun()

            st.markdown("---")

            st.markdown("### 🤖 3. 智慧動態結算 (TWD)")
            balances = {member: 0.0 for member in st.session_state.members}
            total_expense_twd = 0.0

            for exp in st.session_state.expenses:
                amount_twd = exp["折合台幣"]
                payers = exp["_payers_list"]
                consumers = exp["_consumers_list"]
                total_expense_twd += amount_twd

                payer_split = amount_twd / len(payers)
                for p in payers:
                    if p in balances: balances[p] += payer_split

                consumer_split = amount_twd / len(consumers)
                for c in consumers:
                    if c in balances: balances[c] -= consumer_split

            st.info(f"💵 團隊總花費：NT$ {total_expense_twd:,.0f}")

            st.markdown("#### 💰 最終轉帳指示：")
            for member, balance in balances.items():
                if balance > 1:
                    st.success(f"收款方 🟢 **{member}** 可以拿回：NT$ {balance:,.0f}")
                elif balance < -1:
                    st.error(f"付款方 🔴 **{member}** 需要支付：NT$ {abs(balance):,.0f}")
                else:
                    st.write(f"平手 ⚪ **{member}** 不欠錢，完美打平！")

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import datetime, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import requests
import copy
import uuid # 🌟 新增：產生每筆帳單專屬 ID 的套件

# ==========================================
# ⚙️ 網頁初始化設定
# ==========================================
st.set_page_config(page_title="Travel Vibe Pro Max", page_icon="✈️", layout="centered")

# ==========================================
# 🔐 模組零：私有化登入系統
# ==========================================
SECRET_PASSWORD = "201020"

def check_password():
    if "password_correct" not in st.session_state: st.session_state["password_correct"] = False
    if not st.session_state["password_correct"]:
        st.markdown("""
        <div style="background: linear-gradient(135deg, #1f4037, #99f2c8); padding: 30px; border-radius: 10px; text-align: center; color: white; margin-top: 50px;">
            <h1 style="margin: 0; font-size: 36px; font-weight: bold; color: white;">🔒 Travel Vibe 私有終端</h1>
            <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">請輸入通行憑證以解鎖您的專屬行程管家</p>
        </div>
        """, unsafe_allow_html=True)
        password_input = st.text_input("🔑 通關密碼", type="password")
        if st.button("🔓 解鎖進入", use_container_width=True):
            if password_input == SECRET_PASSWORD:
                st.session_state["password_correct"] = True
                st.rerun()
            else: st.error("❌ 密碼錯誤，存取被拒絕！")
        return False
    return True

if check_password():
    # ==========================================
    # 🧠 資料庫與 API 引擎
    # ==========================================
    if 'itinerary' not in st.session_state: st.session_state.itinerary = []
    if 'trip_database' not in st.session_state: st.session_state.trip_database = {}
    if 'search_results' not in st.session_state: st.session_state.search_results = {}
    if 'current_time' not in st.session_state: st.session_state.current_time = datetime.now()
    if 'members' not in st.session_state: st.session_state.members = []
    if 'expenses' not in st.session_state: st.session_state.expenses = []

    geolocator = Nominatim(user_agent="vibe_streamlit_app")
    visa_rules = {
        "🇯🇵 日本": "免簽證 (建議先填 Visit Japan Web)", "🇰🇷 韓國": "免簽證", "🇹🇭 泰國": "需申請簽證/免簽",
        "🇭🇰 香港": "需申請『預辦登記』或『台胞證』", "🇲🇴 澳門": "免簽證", "🇨🇳 中國大陸": "需持有『台胞證』", "🇹🇼 台灣": "身分證/健保卡"
    }

    @st.cache_data(ttl=3600)
    def get_weather(country_name):
        try:
            loc = geolocator.geocode(country_name.split(" ")[1] if " " in country_name else country_name)
            if not loc: return None
            url = f"https://api.open-meteo.com/v1/forecast?latitude={loc.latitude}&longitude={loc.longitude}&current_weather=true"
            res = requests.get(url).json()
            return res['current_weather']
        except: return None

    @st.cache_data(ttl=3600)
    def get_exchange_rates():
        try:
            url = "https://api.exchangerate-api.com/v4/latest/TWD"
            res = requests.get(url).json()
            return res.get("rates", {})
        except:
            return {"TWD": 1.0, "JPY": 4.7, "KRW": 42.0, "USD": 0.031, "EUR": 0.029, "HKD": 0.24, "CNY": 0.22}

    def generate_smart_packing_list(days, country):
        packing = {"🪪 證件與金錢": [], "⛅ 天氣與穿搭推薦": [], "🔌 電子與雜物": []}
        if "台灣" in country: packing["🪪 證件與金錢"].extend(["身分證", "健保卡", "現金 / 信用卡"])
        else:
            packing["🪪 證件與金錢"].extend(["護照 (效期6個月以上)", f"⚠️ 簽證: {visa_rules.get(country, '請查詢')}", "機票/訂房憑證"])

        weather_data = get_weather(country)
        if weather_data:
            temp, code = weather_data['temperature'], weather_data['weathercode']
            is_raining = code >= 50
            packing["⛅ 天氣與穿搭推薦"].append(f"📡 AI 氣象站：氣溫約 {temp}°C，{'🌧️ 預計有雨' if is_raining else '☀️ 天氣晴朗'}")
            if temp < 15: packing["⛅ 天氣與穿搭推薦"].extend(["🧣 發熱衣/保暖內衣", "🧥 羽絨外套或厚大衣"])
            elif temp > 28: packing["⛅ 天氣與穿搭推薦"].extend(["🕶️ 太陽眼鏡", "🧴 防曬乳", "🩳 輕薄透氣短袖"])
            if is_raining: packing["⛅ 天氣與穿搭推薦"].extend(["☔ 摺疊傘 / 輕便雨衣", "🥾 防水鞋套或好乾的鞋子"])
        packing["⛅ 天氣與穿搭推薦"].extend([f"上衣/內外褲 x {days+1} 件", f"襪子 x {days+1} 雙", "好走鞋子"])
        packing["🔌 電子與雜物"].extend(["手機", "充電器/線", "常備藥品", "行動電源"])
        return packing

    def estimate_time(loc1, loc2, mode):
        if not loc1 or not loc2: return 20
        dist_km = geodesic((loc1['lat'], loc1['lng']), (loc2['lat'], loc2['lng'])).km
        speed = 4 if "步行" in mode else 15 if "公車" in mode else 30
        return max(3, int(((dist_km * 1.3) / speed) * 60))

    def format_time_str(mins):
        if mins < 60: return f"{mins} 分"
        return f"{mins // 60} 小時 {mins % 60} 分" if mins % 60 > 0 else f"{mins // 60} 小時"

    st.markdown("""
    <div style="background: linear-gradient(135deg, #8E2DE2, #4A00E0); padding: 20px; border-radius: 10px; text-align: center; color: white; margin-bottom: 20px;">
        <h1 style="margin: 0; font-size: 32px; font-weight: bold; color: white;">✈️ 羅比 普拉思 旅遊用具 </h1>
        <p style="margin: 5px 0 0 0; font-size: 16px; opacity: 0.9;">彈性拆帳 ✕ 行程修改 ✕ 完美規劃 🔒</p>
    </div>
    """, unsafe_allow_html=True)

    if st.sidebar.button("🚪 登出系統"):
        st.session_state["password_correct"] = False
        st.rerun()

    tab_plan, tab_pack, tab_finance = st.tabs(['🗓️ 智慧行程', '🧳 天氣打包', '💸 匯率與拆帳'])

    # ----------------- 🧳 分頁：天氣與打包 -----------------
    with tab_pack:
        col1, col2 = st.columns(2)
        with col1: dest = st.selectbox("✈️ 目的地", list(visa_rules.keys()), index=0)
        with col2: days = st.number_input("📅 天數", min_value=1, max_value=30, value=5)
        if st.button("⛅ 預測天氣並生成清單", use_container_width=True):
            with st.spinner(f"正在連線氣象局..."):
                result = generate_smart_packing_list(days, dest)
                for cat, items in result.items():
                    st.markdown(f"#### {cat}")
                    for item in items: st.markdown(f"- [ ] {item}")
                    st.markdown("---")

    # ----------------- 🗓️ 分頁：行程與檔案庫 -----------------
    with tab_plan:
        with st.expander("📂 歷史行程檔案管理", expanded=False):
            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: trip_name = st.text_input("📁 行程命名", placeholder="例: 東京跨年")
            with c2:
                st.write("")
                if st.button("💾 儲存"):
                    if trip_name and st.session_state.itinerary:
                        st.session_state.trip_database[trip_name] = {
                            "itinerary": copy.deepcopy(st.session_state.itinerary),
                            "members": copy.deepcopy(st.session_state.members),
                            "expenses": copy.deepcopy(st.session_state.expenses)
                        }
                        st.success(f"已儲存：{trip_name}")
            with c3:
                st.write("")
                if st.button("📄 開新"):
                    st.session_state.itinerary = []
                    st.session_state.members = []
                    st.session_state.expenses = []
                    st.rerun()
            if st.session_state.trip_database:
                load_name = st.selectbox("📂 讀取紀錄", list(st.session_state.trip_database.keys()))
                if st.button("📖 讀取此行程"):
                    db_data = st.session_state.trip_database[load_name]
                    st.session_state.itinerary = copy.deepcopy(db_data["itinerary"])
                    st.session_state.members = copy.deepcopy(db_data["members"])
                    st.session_state.expenses = copy.deepcopy(db_data["expenses"])
                    st.success(f"已讀取：{load_name}")
                    st.rerun()

        is_first = len(st.session_state.itinerary) == 0
        if is_first:
            c1, c2 = st.columns(2)
            with c1: start_date = st.date_input("📅 起點日期")
            with c2: start_time = st.time_input("⏰ 出發時間", value=datetime.strptime("09:00", "%H:%M").time())

        c1, c2, c3 = st.columns([1, 2, 1])
        with c1: city = st.text_input("🌍 城市", value="東京")
        with c2: keyword = st.text_input("🔍 找景點", placeholder="例: 成田機場")
        with c3:
            st.write("")
            if st.button("🔎 搜尋", type="primary"):
                with st.spinner("掃描中..."):
                    results = {}
                    try:
                        query = f"{keyword}, {city}" if city else keyword
                        locs = geolocator.geocode(query, exactly_one=False, limit=5)
                        if not locs: locs = geolocator.geocode(keyword, exactly_one=False, limit=5)
                        if locs:
                            for loc in locs: results[loc.address] = {"name": keyword, "lat": loc.latitude, "lng": loc.longitude}
                    except: pass
                    st.session_state.search_results = results

        if st.session_state.search_results:
            selected_address = st.selectbox("📍 選擇正確地點", list(st.session_state.search_results.keys()))
            c1, c2, c3, c4 = st.columns(4)
            with c1: stay = st.number_input("⏱️ 停留(分)", min_value=10, value=120, step=10)
            with c2: mode = st.selectbox("交通", ['🚶 步行', '🚌 公車/地鐵', '🚗 計程車/開車'], index=1)
            with c3: cost = st.number_input("💰 預算", min_value=0, value=0, step=100)
            with c4: note = st.text_input("📝 備註", placeholder="航班號/備註")

            if st.button("➕ 確認加入", use_container_width=True):
                place_data = st.session_state.search_results[selected_address]
                stay_time = timedelta(minutes=stay)
                if is_first:
                    st.session_state.current_time = datetime.combine(start_date, start_time)
                    arrival_time = st.session_state.current_time
                    transport_text = "✨ 出發"
                else:
                    last_stop = st.session_state.itinerary[-1]
                    trans_mins = estimate_time(last_stop['coords'], place_data, mode)
                    arrival_time = st.session_state.current_time + timedelta(minutes=trans_mins)
                    transport_text = f"{mode} (約 {format_time_str(trans_mins)})"

                departure_time = arrival_time + stay_time
                st.session_state.itinerary.append({
                    "arrive": arrival_time.strftime("%m/%d %H:%M"), "transport": transport_text,
                    "name": place_data['name'], "note": note, "stay": format_time_str(stay),
                    "cost": cost, "depart": departure_time.strftime("%m/%d %H:%M"), "coords": place_data
                })
                st.session_state.current_time = departure_time
                st.session_state.search_results = {}
                st.rerun()

        st.markdown("---")
        if st.session_state.itinerary:
            df = pd.DataFrame(st.session_state.itinerary)
            display_plan_df = df[['arrive', 'transport', 'name', 'stay', 'note', 'cost', 'depart']].copy()
            display_plan_df.columns = ['抵達時間', '前往此站', '📍 地點', '⏱️ 停留', '📝 航班/備註', '💰 預算', '離開時間']
            st.dataframe(display_plan_df, use_container_width=True, hide_index=True)
            st.markdown(f"<h3 style='text-align: right; color: #4A00E0;'>💵 總行程預算: NT$ {df['cost'].sum():,.0f} </h3>", unsafe_allow_html=True)

    # ----------------- 💸 分頁：記帳與動態彈性拆帳 -----------------
    with tab_finance:
        st.markdown("### 👥 1. 設定旅伴")
        members_input = st.text_input("請輸入旅伴名字 (用半形逗號 , 分開)", value="Alice, Bob, Kevin")
        if st.button("💾 更新旅伴名單"):
            st.session_state.members = [m.strip() for m in members_input.split(",") if m.strip()]
            st.success(f"已更新旅伴：{', '.join(st.session_state.members)}")

        st.markdown("---")

        st.markdown("### 💱 2. 新增帳單")
        if not st.session_state.members:
            st.warning("⚠️ 請先在上方輸入旅伴名字！")
        else:
            rates = get_exchange_rates()
            currency_options = {"🇹🇼 台幣 (TWD)": "TWD", "🇯🇵 日幣 (JPY)": "JPY", "🇰🇷 韓元 (KRW)": "KRW", "🇺🇸 美金 (USD)": "USD"}

            c1, c2, c3 = st.columns([2, 1, 1])
            with c1: exp_item = st.text_input("消費項目", placeholder="例: 敘敘苑燒肉、和服體驗")
            with c2:
                curr_label = st.selectbox("幣別", list(currency_options.keys()))
                curr_code = currency_options[curr_label]
            with c3: exp_amount = st.number_input("外幣金額", min_value=0.0, step=100.0)

            c4, c5 = st.columns(2)
            with c4: exp_payers = st.multiselect("💳 誰付的錢？(可多選)", st.session_state.members, default=st.session_state.members[0:1] if st.session_state.members else [])
            with c5: exp_consumers = st.multiselect("🍽️ 分攤給誰？(打叉可刪除)", st.session_state.members, default=st.session_state.members)

            if curr_code in rates and rates[curr_code] > 0: twd_estimate = exp_amount / rates[curr_code]
            else: twd_estimate = exp_amount

            st.caption(f"💡 即時匯率換算：約折合 **NT$ {twd_estimate:,.0f}**")

            if st.button("➕ 新增這筆帳款", type="primary"):
                if not exp_item or exp_amount <= 0: st.error("❌ 請輸入項目名稱與大於 0 的金額！")
                elif not exp_payers: st.error("❌ 至少要選一個付款人！")
                elif not exp_consumers: st.error("❌ 至少要有一個人來分攤這筆費用！")
                else:
                    # 🌟 賦予這筆帳單一個獨一無二的 ID (身分證)
                    st.session_state.expenses.append({
                        "id": str(uuid.uuid4()),
                        "項目": exp_item,
                        "原幣別": curr_code,
                        "_original_amount": exp_amount, # 紀錄原始輸入的金額，方便之後修改
                        "折合台幣": round(twd_estimate),
                        "付款人": ", ".join(exp_payers),
                        "分攤給": ", ".join(exp_consumers),
                        "_payers_list": exp_payers,
                        "_consumers_list": exp_consumers
                    })
                    st.success(f"✅ 記帳成功！")
                    st.rerun()

        # 📋 顯示明細與修改區塊
        if st.session_state.expenses:
            st.markdown("#### 📋 帳單明細")
            # 為了避免舊資料沒有 id 產生錯誤，自動補上 id
            for exp in st.session_state.expenses:
                if "id" not in exp: exp["id"] = str(uuid.uuid4())
                if "_original_amount" not in exp: exp["_original_amount"] = float(exp["折合台幣"])

            exp_df = pd.DataFrame(st.session_state.expenses)
            display_exp_df = exp_df[["項目", "原幣別", "折合台幣", "付款人", "分攤給"]]
            st.dataframe(display_exp_df, use_container_width=True, hide_index=True)

            # 🌟 全新編輯與刪除面板
            with st.expander("✏️ 修改或刪除單筆帳款", expanded=False):
                # 建立下拉選單選項：項目名稱 (折合台幣)
                exp_dict = {exp["id"]: exp for exp in st.session_state.expenses}
                def format_exp(exp): return f"{exp['項目']} (NT$ {exp['折合台幣']})"

                selected_id = st.selectbox("選擇要修改的帳單：", list(exp_dict.keys()), format_func=lambda x: format_exp(exp_dict[x]))

                if selected_id:
                    sel_exp = exp_dict[selected_id]

                    st.write("重新設定此筆帳款資料：")
                    e1, e2, e3 = st.columns([2, 1, 1])
                    with e1: e_item = st.text_input("新項目名稱", value=sel_exp["項目"], key="e_item")
                    with e2:
                        idx_curr = list(currency_options.values()).index(sel_exp["原幣別"])
                        e_curr_label = st.selectbox("新幣別", list(currency_options.keys()), index=idx_curr, key="e_curr")
                        e_curr_code = currency_options[e_curr_label]
                    with e3: e_amount = st.number_input("新外幣金額", value=float(sel_exp["_original_amount"]), step=100.0, key="e_amount")

                    e4, e5 = st.columns(2)
                    with e4: e_payers = st.multiselect("💳 新付款人", st.session_state.members, default=sel_exp["_payers_list"], key="e_payers")
                    with e5: e_consumers = st.multiselect("🍽️ 新分攤給", st.session_state.members, default=sel_exp["_consumers_list"], key="e_consumers")

                    if e_curr_code in rates and rates[e_curr_code] > 0: e_twd = e_amount / rates[e_curr_code]
                    else: e_twd = e_amount

                    st.caption(f"💡 修改後重新換算：約折合 **NT$ {e_twd:,.0f}**")

                    b1, b2 = st.columns(2)
                    with b1:
                        if st.button("💾 儲存修改", use_container_width=True):
                            if not e_item or e_amount <= 0 or not e_payers or not e_consumers:
                                st.error("❌ 欄位填寫不完整！")
                            else:
                                for i, exp in enumerate(st.session_state.expenses):
                                    if exp["id"] == selected_id:
                                        st.session_state.expenses[i].update({
                                            "項目": e_item, "原幣別": e_curr_code, "_original_amount": e_amount,
                                            "折合台幣": round(e_twd), "付款人": ", ".join(e_payers),
                                            "分攤給": ", ".join(e_consumers), "_payers_list": e_payers, "_consumers_list": e_consumers
                                        })
                                        break
                                st.success("✅ 修改成功！")
                                st.rerun()
                    with b2:
                        if st.button("🗑️ 刪除此筆", use_container_width=True):
                            st.session_state.expenses = [exp for exp in st.session_state.expenses if exp["id"] != selected_id]
                            st.success("✅ 刪除成功！")
                            st.rerun()

            if st.button("🗑️ 清空所有帳單", type="secondary"):
                st.session_state.expenses = []
                st.rerun()

            st.markdown("---")

            st.markdown("### 🤖 3. 智慧動態結算 (TWD)")
            balances = {member: 0.0 for member in st.session_state.members}
            total_expense_twd = 0.0

            for exp in st.session_state.expenses:
                amount_twd = exp["折合台幣"]
                payers = exp["_payers_list"]
                consumers = exp["_consumers_list"]
                total_expense_twd += amount_twd

                payer_split = amount_twd / len(payers)
                for p in payers:
                    if p in balances: balances[p] += payer_split

                consumer_split = amount_twd / len(consumers)
                for c in consumers:
                    if c in balances: balances[c] -= consumer_split

            st.info(f"💵 團隊總花費：NT$ {total_expense_twd:,.0f}")

            st.markdown("#### 💰 最終轉帳指示：")
            for member, balance in balances.items():
                if balance > 1:
                    st.success(f"收款方 🟢 **{member}** 可以拿回：NT$ {balance:,.0f}")
                elif balance < -1:
                    st.error(f"付款方 🔴 **{member}** 需要支付：NT$ {abs(balance):,.0f}")
                else:
                    st.write(f"平手 ⚪ **{member}** 不欠錢，完美打平！")